In [1]:
# Importing libraries
import xarray as xr
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
import os
import glob
import cftime
import time
import re
import multiprocessing



# Importing functions
import Functions.grid_func as grid_func
import Functions.wind_model_func as wind_model_func

/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [ ]:

variables = ['U100', 'V100', 'SSRD', 'tas', 'tmax'] # List of variables
path = '/climca/people/ppfleiderer/ERA5/RL_climate/ERA5_raw/*'

# Select all files in the range 2015-2024
year_range = (1960, 2024)
all_files = [f for f in glob.glob(path) if f.endswith('.nc')]
all_files = sorted(all_files)

filtered_files = []
for file in all_files:
    # Extract year and month using regex
    match = re.search(r'(\d{4})', file)
    if match:
        year = int(match.group(1))
        if year_range is None or year_range[0] <= year <= year_range[1]:
            filtered_files.append(file)

# filtered_files
print(f'Found {len(filtered_files)} files for variables {variables} in the specified year range {year_range}.')
# Read datasets for each variable
files_by_variable = {}

# Group files by variable name
for f in filtered_files:
    match = re.search(r'/([^/]+)_(\d{4})\.nc$', f)
    if match:
        var = match.group(1)
        if var not in files_by_variable:
            files_by_variable[var] = []
        files_by_variable[var].append(f)

files_by_variable
# SSRD_list = files_by_variable['SSRD']
# SSRD_list

# Read datasets for each variable
datasets_by_variable = {}
for var, files in files_by_variable.items():
    print(f'Processing {var}...')
    files_sorted = sorted(files)

    # Use the first file as coordinate reference
    ref_ds = xr.open_dataset(files_sorted[0])
    ref_lat = ref_ds.lat
    ref_lon = ref_ds.lon

    def preprocess(ds):
        ds = ds.sortby('lat')  # Ensure consistent order
        ds = ds.assign_coords(lat=ref_lat, lon=ref_lon)  # Align coordinates exactly
        return ds

    # Open and process all datasets with aligned coordinates
    ds = xr.open_mfdataset(files_sorted, combine='by_coords', preprocess=preprocess)
    
    if var == 'SSRD':
        ds = ds / 3600 # Convert from J/m2 to W/m2
    ds_daily = ds.resample(time='1D').mean()
    datasets_by_variable[var] = ds_daily
    if var == 'tas':
        tasmax = ds.resample(time='1D').max()
        tasmax = tasmax.rename({'var167': 'tasmax'})
        datasets_by_variable['tasmax'] = tasmax


datasets_by_variable

var_names = {'var169': 'rsds',
             'var167': 'tas',
             'var246': 'U100',
             'var247': 'V100',
             'tasmax': 'tasmax'}

ds_list = []

for i in datasets_by_variable:
    # print(i)
    print(datasets_by_variable[i])
    # print('------------------')
    ds = datasets_by_variable[i]
    var = list(ds.data_vars)[0]
    ds = ds.rename({var: var_names[var]})
    ds = ds.sel(lat=slice(45, 60), lon=slice(4, 17))
    nc = grid_func.regrid(ds, s = 47, n = 56, w = 6, e = 16) # One ° less in the north to prevent NaN values
    # Append to list for later merging
    ds_list.append(nc)
    # ds.to_netcdf(f'/climca/people/onennecke/ERA5/{i}.nc')

# Read in tmax

# Combine all into a single dataset
ERA5_ds = xr.merge(ds_list)

# Assign coordinates for ESM
ERA5_ds = ERA5_ds.assign_coords(ESM='ERA5')
ERA5_ds = ERA5_ds.assign_coords(run='hist')  # Assign run coordinate
ERA5_ds = ERA5_ds.assign_coords(ESM_run='ERA5_hist')  # Assign ESM_run coordinate

# Remove every 29.02
ERA5_ds = ERA5_ds.where(~((ERA5_ds['time.month'] == 2) & (ERA5_ds['time.day'] == 29)), drop=True)

ERA5_ds['sfcWind'] = np.sqrt(ERA5_ds['U100']**2 + ERA5_ds['V100']**2)
# ERA5_ds['tas'] = ERA5_ds['tas'] - 273.15
# ERA5_ds['tasmax'] = ERA5_ds['tasmax'] - 273.15

ERA5_ds

ERA5_ssrd =  ERA5_ds['rsds']
# Save ERA5 data to a netCDF file
output_file = '/climca/people/onennecke/not_debiased_data/ERA5_1960_2024/ERA5_hist_rsds.nc'
if os.path.isfile(output_file) == False:
    ERA5_ssrd.to_netcdf(output_file) 
# ERA5_ssrd
ERA5_sfcWind = ERA5_ds['sfcWind']
# Save ERA5 data to a netCDF file
output_file = '/climca/people/onennecke/not_debiased_data/ERA5_1960_2024/ERA5_hist_sfcWind.nc'
if os.path.isfile(output_file) == False:
    ERA5_sfcWind.to_netcdf(output_file)
# ERA5_sfcWind
ERA5_tas = ERA5_ds['tas']
# Save ERA5 data to a netCDF file
output_file = '/climca/people/onennecke/not_debiased_data/ERA5_1960_2024/ERA5_hist_tas.nc'
if os.path.isfile(output_file) == False:
    ERA5_tas.to_netcdf(output_file)
# ERA5_tas
ERA5_tasmax = ERA5_ds['tasmax']
# Save ERA5 data to a netCDF file
output_file = '/climca/people/onennecke/not_debiased_data/ERA5_1960_2024/ERA5_hist_tasmax.nc'
if os.path.isfile(output_file) == False:
    ERA5_tasmax.to_netcdf(output_file)
# ERA5_tasmax

ERA5_time = ERA5_ds['time'].load()
print('Vars loaded, now processing psl...')

variable = 'slp'
# Select all files in the range 2014-2024
path = f'/climca/data/ERA5/daily/{variable}/'
year_range = (1960, 2024)
all_files = sorted(glob.glob(os.path.join(path, '*.nc')))

filtered_files = []
for file in all_files:
    # Extract year and month using regex
    match = re.search(r'(\d{4})', file)
    if match:
        year = int(match.group(1))
        if year_range is None or year_range[0] <= year <= year_range[1]:
            filtered_files.append(file)

filtered_files

ds = xr.open_mfdataset(filtered_files, combine='by_coords', preprocess=grid_func.preprocess_ERA5_psl)

# Regrid the dataset
regridded_ds = grid_func.regrid(ds, s = 30, n = 70, w = 340, e = 30)

# Assign coordinates for ESM
regridded_ds = regridded_ds.assign_coords(ESM='ERA5')
regridded_ds = regridded_ds.assign_coords(run='hist')  # Assign run coordinate
regridded_ds = regridded_ds.assign_coords(ESM_run='ERA5_hist')  # Assign ESM_run coordinate


# Rename the variable to 'pls'
ERA5_ds_psl = regridded_ds.rename({'var151': 'psl'})

ERA5_ds_psl = ERA5_ds_psl.where(~((ERA5_ds_psl['time.month'] == 2) & (ERA5_ds_psl['time.day'] == 29)), drop=True)

ERA5_ds_psl = ERA5_ds_psl.assign_coords(time=ERA5_time)  # Ensure time coordinates are aligned

# Save ERA5 data to a netCDF file
output_file = '/climca/people/onennecke/not_debiased_data/ERA5_1960_2024/ERA5_hist_psl.nc'
if os.path.isfile(output_file) == False:
    ERA5_ds_psl.to_netcdf(output_file) 


Found 260 files for variables ['U100', 'V100', 'SSRD', 'tas', 'tmax'] in the specified year range (1960, 2024).
Processing SSRD...
Processing U100...
Processing V100...
Processing tas...
<xarray.Dataset> Size: 165MB
Dimensions:  (time: 23742, lat: 37, lon: 47)
Coordinates:
  * lat      (lat) float64 296B 45.15 45.45 45.75 46.05 ... 55.35 55.65 55.95
  * lon      (lon) float64 376B 3.0 3.3 3.6 3.9 4.2 ... 15.6 15.9 16.2 16.5 16.8
  * time     (time) datetime64[ns] 190kB 1960-01-01 1960-01-02 ... 2024-12-31
Data variables:
    var169   (time, lat, lon) float32 165MB dask.array<chunksize=(1, 37, 47), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.2.4 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    institution:  European Centre for Medium-Range Weather Forecasts
    history:      Tue Aug 06 16:01:40 2024: cdo -f nc -remapnn,U100_2023.nc S...
    CDO:          Climate Data Operators version 2.2.2 (https://mpimet.mpg.de...
<xarray.Dataset> Size: 165MB
